# DocTrack — Générer la voix wolof de l'assistant (xTTS-v2-wolof, GalsenAI)

L'assistant utilise toujours les mêmes phrases (« Ban kayit moo la réer ? »…). On les fait lire **une seule fois** ici,
avec la carte graphique gratuite de Colab, puis on dépose les fichiers MP3 dans DocTrack : la voix est ensuite instantanée et gratuite.

**Avant de commencer**
1. **Exécution → Modifier le type d'exécution → GPU T4**.
2. Sur votre PC, dans le dossier de l'API : `php artisan doctrack:assistant-phrases --lang=wo`
   → fichier `storage/app/assistant/phrases_wo.json` à envoyer à l'étape 3.
3. (Facultatif) un enregistrement **WAV de 6 à 15 secondes** d'une voix claire, sans bruit, **avec l'accord de la personne** :
   le modèle imitera cette voix. Sans fichier, on utilise la voix d'exemple fournie avec le modèle.

**Licence** : modèle dérivé de Coqui XTTS-v2 (licence *Coqui Public Model License*, **non commerciale**).
Utilisable pour la démonstration et le pilote gratuit ; à remplacer ou à faire autoriser avant une exploitation payante.

⚠️ Première utilisation : cellule à valider ensemble (les dépôts du modèle évoluent). En cas d'erreur, copiez le message à Claude.

## 1. Installation (5 à 10 minutes)

In [ ]:
!git clone -q --depth 1 https://github.com/Galsenaicommunity/Wolof-TTS.git
!pip -q install coqui-tts huggingface_hub pydub
!apt-get -qq install -y ffmpeg > /dev/null
import sys, torch
# Le dépôt GalsenAI contient une version adaptée de la bibliothèque TTS : on la place en priorité.
sys.path.insert(0, '/content/Wolof-TTS/notebooks/Models/xTTS v2')
print('GPU :', torch.cuda.is_available())

## 2. Télécharger le modèle wolof (≈ 7 Go)

In [ ]:
import os, glob
from huggingface_hub import snapshot_download
model_dir = snapshot_download('galsenai/xTTS-v2-wolof')
fichiers = sorted(glob.glob(model_dir + '/**/*', recursive=True))
for f in fichiers:
    if os.path.isfile(f): print(round(os.path.getsize(f)/1e6, 1), 'Mo', f.replace(model_dir, ''))

def trouve(motif, obligatoire=True):
    res = [f for f in fichiers if glob.fnmatch.fnmatch(os.path.basename(f), motif)]
    if not res and obligatoire: raise FileNotFoundError(motif)
    return sorted(res, key=os.path.getsize, reverse=True)[0] if res else None

checkpoint = trouve('*.pth')                 # le plus gros .pth = poids du modèle
config_json = trouve('config.json')
vocab_json  = trouve('vocab.json')
speakers    = trouve('speakers_xtts.pth', obligatoire=False)
echantillon = trouve('*.wav', obligatoire=False)
print('\nmodèle :', checkpoint, '\nconfig :', config_json, '\nvocab :', vocab_json, '\nvoix d\'exemple :', echantillon)

## 3. Charger le modèle

In [ ]:
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
config = XttsConfig(); config.load_json(config_json)
model = Xtts.init_from_config(config)
kwargs = dict(checkpoint_path=checkpoint, vocab_path=vocab_json, use_deepspeed=False)
if speakers: kwargs['speaker_file_path'] = speakers
model.load_checkpoint(config, **kwargs)
model.to(device)
print('Modèle chargé ✔')

## 4. Envoyer `phrases_wo.json` (et, si vous l'avez, la voix de référence .wav)

In [ ]:
from google.colab import files
import json
up = files.upload()
phrases = json.load(open(next(n for n in up if n.endswith('.json')), encoding='utf-8'))
voix = next((n for n in up if n.lower().endswith('.wav')), None) or echantillon
print(len(phrases), 'phrases — voix de référence :', voix)
gpt_cond_latent, speaker_embedding = model.get_conditioning_latents(
    audio_path=[voix], gpt_cond_len=model.config.gpt_cond_len,
    max_ref_length=model.config.max_ref_len, sound_norm_refs=model.config.sound_norm_refs)

## 5. Essai sur une phrase (écoutez avant de tout générer)

In [ ]:
import numpy as np, soundfile as sf
from IPython.display import Audio, display
def lire(texte):
    texte = texte.lower().replace('docTrack'.lower(), 'dok trak')   # prononciation du nom
    for langue in ('wo', 'en'):
        try:
            out = model.inference(text=texte, language=langue, gpt_cond_latent=gpt_cond_latent,
                                  speaker_embedding=speaker_embedding, temperature=0.7,
                                  repetition_penalty=10.0, speed=1.0, enable_text_splitting=True)
            return np.asarray(out['wav'], dtype=np.float32)
        except Exception as e:
            erreur = e
    raise erreur
essai = lire(phrases[0]['text'])
display(Audio(essai, rate=24000))

## 6. Générer toutes les phrases → MP3 (silences retirés)

In [ ]:
from pydub import AudioSegment, silence
os.makedirs('audio_wo', exist_ok=True)
def nettoie(wav_path, mp3_path):
    son = AudioSegment.from_wav(wav_path)
    morceaux = silence.split_on_silence(son, min_silence_len=450, silence_thresh=son.dBFS - 16, keep_silence=180)
    propre = sum(morceaux, AudioSegment.silent(0)) if morceaux else son
    propre.export(mp3_path, format='mp3', bitrate='64k')
for i, p in enumerate(phrases, 1):
    wav = f"audio_wo/{p['key']}.wav"
    sf.write(wav, lire(p['text']), 24000)
    nettoie(wav, f"audio_wo/{p['key']}.mp3"); os.remove(wav)
    print(f"{i}/{len(phrases)}", p['key'])

## 7. Écouter quelques résultats puis télécharger le ZIP

In [ ]:
for p in phrases[:3]:
    print(p['key'], '—', p['text']); display(Audio(f"audio_wo/{p['key']}.mp3"))
!cd audio_wo && zip -q -r ../audio_wo.zip *.mp3
files.download('audio_wo.zip')

## 8. Installer les fichiers dans DocTrack

Décompressez `audio_wo.zip` dans le dossier de l'API : **`storage/app/assistant/audio/wo/`**
(un fichier par phrase : `welcome.mp3`, `ask_doc_type_lost.mp3`…).

Vérification : `GET /api/v2/assistant/capabilities` → `audio_phrases.wo` doit afficher le nombre de fichiers.

Faites écouter les phrases à des locuteurs wolof : si une phrase sonne mal, corrigez son texte dans `config/assistant.php`,
relancez l'export avec `--missing` après avoir supprimé l'ancien MP3, et régénérez seulement celle-là.